In [ ]:
import sys; sys.path.append('..')
import sheet_convergence, sim_utils
import MeshFEM
from tri_mesh_viewer import TriMeshViewer
import numpy as np

import meshing, mesh, elastic_solid, energy, benchmark
import fd_validation

pts, _ = sheet_convergence.stripBoundary(1)
m = mesh.Mesh(*meshing.tetrahedralize_extruded_polylines([np.array(pts + [pts[0]])], [], thickness=4.0, maxVol=0.001), degree=1)
es = elastic_solid.ElasticSolid(m, energy.NeoHookeanYoungPoisson(3, 1, 0))

es_re = elastic_solid.ElasticSolidRotExtrap(m, energy.NeoHookeanYoungPoisson(3, 1, 0))

In [ ]:
x_rest = es.getVars().copy()
x = x_rest.copy()

In [ ]:
minZVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_Z)
maxZVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_Z)

x[minZVars[2::3]] *= 1.05
x[maxZVars[2::3]] *= 1.05

es.setVars(x)
es.computeEquilibrium([], fixedVars=(minZVars + maxZVars))

In [ ]:
x_orig = es.getVars().copy()

In [ ]:
es_re.setVars(x_orig)
es_re.setVars(x_orig + 1e-3 * np.random.normal(size=x_orig.shape))

In [ ]:
benchmark.reset()
fd_validation.gradConvergencePlot(es_re)
benchmark.report()

In [ ]:
# This plot should look wrong since the source config is not up-to-date
benchmark.reset()
fd_validation.gradConvergencePlot(es_re, customArgs={'updatedParametrization': True})
benchmark.report()

In [ ]:
# This plot should look wrong since the source config is not up-to-date
benchmark.reset()
fd_validation.hessConvergencePlot(es_re)
benchmark.report()

In [ ]:
# Hessian can only be evaluated with an up-to-date source config
es_re.updateParametrization()

In [ ]:
benchmark.reset()
fd_validation.hessConvergencePlot(es_re)
benchmark.report()